# 🧠 AI Trinh Sát: Deep Learning Bi-LSTM (Ultimate Local Data Fusion)

**Mục tiêu:** Quét toàn bộ dữ liệu từ ổ cứng Local (Ổ D:), dung hợp các nguồn Kaggle (XSS, Command Injection, JSON Web Payloads) và HttpParamsDataset để tạo ra lõi WAF tối thượng.

## 📦 1. Import Thư viện

In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import json
import re
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print("🚀 Khởi động Hệ thống Deep Learning Bi-LSTM (Ultimate Local Data Fusion)...")

🚀 Khởi động Hệ thống Deep Learning Bi-LSTM (Ultimate Local Data Fusion)...


## 📂 2. Siêu máy hút bụi dữ liệu từ Ổ D:

In [18]:
print("📂 Đang tiến hành hút và dung hợp dữ liệu...")
dfs = []

# --- A. Xử lý nhóm file HttpParamsDataset ---
def normalize_http_label(label):
    label = str(label).lower().strip()
    if label == 'norm': return 'Normal'
    if 'xss' in label or 'js-syntax' in label: return 'XSS'  # Gộp JS-Syntax vào XSS
    if 'sql' in label: return 'SQLi'
    if 'cmd' in label or 'exec' in label: return 'Command Injection'
    if 'path' in label or 'traversal' in label: return 'Path Traversal'
    if 'ssrf' in label: return 'SSRF'
    return label.upper()

http_files = [
    r"D:\AI\clawweb\data\httpparagram\payload_train.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test_lexical.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_full.csv"
]

for f in http_files:
    if os.path.exists(f):
        df_temp = pd.read_csv(f)
        df_temp = df_temp[['payload', 'attack_type']].rename(columns={'payload': 'text', 'attack_type': 'label'})
        df_temp['label'] = df_temp['label'].apply(normalize_http_label)
        dfs.append(df_temp)
        print(f"✅ Đã nạp {len(df_temp)} dòng từ {os.path.basename(f)}")
    else:
        print(f"⚠️ Không tìm thấy: {f}")

# --- B. Xử lý bộ XSS Kaggle ---
path_xss = r"D:\AI\clawweb\data\xss\XSS_dataset.csv"
if os.path.exists(path_xss):
    df_xss = pd.read_csv(path_xss)
    df_xss = df_xss[['Sentence', 'Label']].rename(columns={'Sentence': 'text', 'Label': 'label'})
    df_xss['label'] = df_xss['label'].map({1: 'XSS', 0: 'Normal'})
    dfs.append(df_xss)
    print(f"✅ Đã nạp {len(df_xss)} dòng từ XSS_dataset.csv")

# --- C. Xử lý bộ Command Injection Kaggle ---
path_cmd = r"D:\AI\clawweb\data\oscommand\command injection.csv"
if os.path.exists(path_cmd):
    df_cmd = pd.read_csv(path_cmd)
    df_cmd = df_cmd[['sentence', 'Label']].rename(columns={'sentence': 'text', 'Label': 'label'})
    df_cmd['label'] = df_cmd['label'].map({1: 'Command Injection', 0: 'Normal'})
    dfs.append(df_cmd)
    print(f"✅ Đã nạp {len(df_cmd)} dòng từ command injection.csv")

# --- D. Xử lý bộ SQL Injection (Dự đoán đường dẫn) ---
path_sqli = r"D:\AI\clawweb\data\sqli\Modified_SQL_Dataset.csv"
if not os.path.exists(path_sqli):
    path_sqli = r"D:\AI\clawweb\data\sql_injection\Modified_SQL_Dataset.csv"

if os.path.exists(path_sqli):
    df_sqli = pd.read_csv(path_sqli)
    df_sqli = df_sqli.rename(columns={"Query": "text", "Label": "label"})[["text", "label"]]
    df_sqli["label"] = df_sqli["label"].map({1: "SQLi", 0: "Normal", "1": "SQLi", "0": "Normal"})
    dfs.append(df_sqli)
    print(f"✅ Đã nạp {len(df_sqli)} dòng từ Modified_SQL_Dataset.csv")

# --- E. Xử lý JSONL (Dùng Regex Bất Tử chống lỗi file Kaggle) ---
path_jsonl = r"D:\AI\clawweb\data\web_payloads\WEB_APPLICATION_PAYLOADS.jsonl"
if os.path.exists(path_jsonl):
    json_payloads = []
    with open(path_jsonl, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Dùng Biểu thức chính quy (Regex) để cào dữ liệu thay vì json.loads
    # Việc này giúp bỏ qua hoàn toàn các dòng bị lỗi thiếu dấu phẩy của Kaggle
    matches = re.finditer(r'"payload"\s*:\s*"(.*?)"[\s\S]*?"type"\s*:\s*"(.*?)"', content)
    for match in matches:
        payload_text = match.group(1).replace('\\"', '"')
        payload_type = match.group(2).upper()
        if 'SSRF' in payload_type:
            json_payloads.append((payload_text, 'SSRF'))
        elif 'CSRF' in payload_type:
            json_payloads.append((payload_text, 'CSRF'))

    if json_payloads:
        df_json = pd.DataFrame(json_payloads, columns=["text", "label"])
        df_json = pd.concat([df_json] * 50, ignore_index=True) # Nhân bản lên 50 lần
        dfs.append(df_json)
        print(f"✅ Đã cào & nhân bản {len(df_json)} dòng SSRF/CSRF bằng Regex Engine")
    else:
        print("⚠️ Không trích xuất được SSRF/CSRF nào từ Regex.")

# --- GOM TẤT CẢ VÀ DỌN DẸP ---
if not dfs:
    raise ValueError("❌ LỖI NGHIÊM TRỌNG: Không đọc được bất kỳ file nào từ ổ D:!")

df = pd.concat(dfs, ignore_index=True)
df['text'] = df['text'].astype(str)

initial_len = len(df)
df = df.drop_duplicates().dropna() # Xóa sạch rác trùng lặp
print(f"\n🧹 Đã dọn dẹp {initial_len - len(df)} dòng copy/paste trùng lặp.")
print(f"📊 Tổng lực lượng Data hiện có: {len(df)} dòng.")
print("\n🔥 Phân bổ Các Nhãn Lỗ Hổng (Đã Sạch Sẽ):")
print(df['label'].value_counts())

📂 Đang tiến hành hút và dung hợp dữ liệu...
✅ Đã nạp 20712 dòng từ payload_train.csv
✅ Đã nạp 10355 dòng từ payload_test.csv
✅ Đã nạp 1106 dòng từ payload_test_lexical.csv
✅ Đã nạp 31067 dòng từ payload_full.csv
✅ Đã nạp 13686 dòng từ XSS_dataset.csv
✅ Đã nạp 2106 dòng từ command injection.csv
✅ Đã nạp 30919 dòng từ Modified_SQL_Dataset.csv
✅ Đã cào & nhân bản 9950 dòng SSRF/CSRF bằng Regex Engine

🧹 Đã dọn dẹp 52753 dòng copy/paste trùng lặp.
📊 Tổng lực lượng Data hiện có: 67148 dòng.

🔥 Phân bổ Các Nhãn Lỗ Hổng (Đã Sạch Sẽ):
label
Normal               36359
SQLi                 21929
XSS                   7873
Command Injection      520
Path Traversal         290
CSRF                    92
SSRF                    85
Name: count, dtype: int64


## ⚖️ 3. Xử lý Lệch Dữ Liệu (Undersampling)

In [19]:
print("\n⚖️ Đang cân bằng lại dữ liệu (Undersampling nhóm Normal)...")
df_normal = df[df['label'] == 'Normal']
df_attack = df[df['label'] != 'Normal']

total_attacks = len(df_attack)

if len(df_normal) > total_attacks * 1.5:
    df_normal = df_normal.sample(n=int(total_attacks * 1.5), random_state=42)
    print(f"🔪 Đã chặt bớt Normal xuống còn {len(df_normal)} dòng để AI bớt lười.")

df = pd.concat([df_normal, df_attack]).sample(frac=1, random_state=42).reset_index(drop=True)
print("\n📊 Tỷ lệ dữ liệu sau khi cắt tỉa:")
print(df['label'].value_counts())


⚖️ Đang cân bằng lại dữ liệu (Undersampling nhóm Normal)...

📊 Tỷ lệ dữ liệu sau khi cắt tỉa:
label
Normal               36359
SQLi                 21929
XSS                   7873
Command Injection      520
Path Traversal         290
CSRF                    92
SSRF                    85
Name: count, dtype: int64


## 🧠 4. Tiền xử lý & Chia tập (Tokenization)

In [20]:
le = LabelEncoder()
y = le.fit_transform(df['label'])
num_classes = len(le.classes_)

print("🧠 Đang xây dựng từ điển nhúng ký tự (Tokenizer)...")
MAX_WORDS = 10000  
MAX_LEN = 150      

tokenizer = Tokenizer(num_words=MAX_WORDS, char_level=True, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])

X_pad = pad_sequences(tokenizer.texts_to_sequences(df['text']), maxlen=MAX_LEN, padding='post', truncating='post')

X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42)
print(f"Tập train: {X_train.shape}, Tập test: {X_test.shape}")

🧠 Đang xây dựng từ điển nhúng ký tự (Tokenizer)...
Tập train: (53718, 150), Tập test: (13430, 150)


## 🏗️ 5. Xây dựng Mạng Nơ-ron (Bi-LSTM)

In [21]:
print("🏗️ Đang lắp ráp bộ não nơ-ron Bi-LSTM...")
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5), 
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

🏗️ Đang lắp ráp bộ não nơ-ron Bi-LSTM...


c:\Users\truon\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 🔥 6. Huấn luyện (Training)

In [22]:
print("⚖️ Đang tính toán trọng số động (Class Weights)...")
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

print("\n🔥 Bắt đầu quá trình Huấn luyện (Training)...")
history = model.fit(
    X_train, y_train, 
    epochs=5, 
    batch_size=128, 
    validation_data=(X_test, y_test),
    class_weight=class_weights_dict
)

⚖️ Đang tính toán trọng số động (Class Weights)...

🔥 Bắt đầu quá trình Huấn luyện (Training)...
Epoch 1/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 119s 263ms/step - accuracy: 0.5918 - loss: 1.0716 - val_accuracy: 0.9188 - val_loss: 0.4517
Epoch 2/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 105s 250ms/step - accuracy: 0.8531 - loss: 0.4769 - val_accuracy: 0.9598 - val_loss: 0.1814
Epoch 3/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 98s 233ms/step - accuracy: 0.9167 - loss: 0.3020 - val_accuracy: 0.9656 - val_loss: 0.1538
Epoch 4/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 95s 227ms/step - accuracy: 0.9411 - loss: 0.1963 - val_accuracy: 0.9415 - val_loss: 0.2055
Epoch 5/5
420/420 ━━━━━━━━━━━━━━━━━━━━ 98s 233ms/step - accuracy: 0.9486 - loss: 0.1691 - val_accuracy: 0.9666 - val_loss: 0.1346


## 💾 7. Lưu Mô hình (Xuất ra Ổ D:)

In [23]:
print("\n💾 Đang xuất file não...")
save_dir = r"D:\AI\clawweb\model"
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, 'deep_learning_agent_core.keras')) 

with open(os.path.join(save_dir, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(save_dir, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

print(f"✅ HOÀN TẤT! Sẵn sàng mang não này đi đục Web. Files đã được lưu tại: {save_dir}")


💾 Đang xuất file não...
✅ HOÀN TẤT! Sẵn sàng mang não này đi đục Web. Files đã được lưu tại: D:\AI\clawweb\model


## 🧪 8. Chạy Thử Nghiệm Thực Tế

In [ ]:
def test_ai(payload):
    seq = tokenizer.texts_to_sequences([str(payload)])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred_probs = model.predict(pad, verbose=0)[0]
    pred_class = np.argmax(pred_probs)
    label = le.inverse_transform([pred_class])[0]
    confidence = pred_probs[pred_class] * 100
    print(f"Payload: {payload}\n=> Phát hiện: [ {label} ] (Độ tự tin: {confidence:.2f}%)\n")

print("--- 🛡️ TEST HACKER ---")
test_ai("admin' OR 1=1 --")
test_ai("<img src='x' onerror='alert(1)'>")
test_ai("test && cat /etc/passwd")
test_ai("../../../../etc/shadow")
test_ai("http://169.254.169.254/latest/meta-data/")

print("--- 🟢 TEST NORMAL ---")
test_ai("Xin chào, tôi muốn tìm tài liệu NCKH")
test_ai("https://www.google.com/search?q=cat")

--- 🛡️ TEST HACKER ---
Payload: admin' OR 1=1 --
=> Phát hiện: [ Command Injection ] (Độ tự tin: 74.67%)

Payload: <img src='x' onerror='alert(1)'>
=> Phát hiện: [ XSS ] (Độ tự tin: 93.02%)

Payload: test && cat /etc/passwd
=> Phát hiện: [ Command Injection ] (Độ tự tin: 99.96%)

Payload: ../../../../etc/shadow
=> Phát hiện: [ Path Traversal ] (Độ tự tin: 99.98%)

Payload: http://169.254.169.254/latest/meta-data/
=> Phát hiện: [ SSRF ] (Độ tự tin: 99.93%)

--- 🟢 TEST NORMAL ---
Payload: Xin chào, tôi muốn tìm tài liệu NCKH
=> Phát hiện: [ Normal ] (Độ tự tin: 96.92%)

Payload: https://www.google.com/search?q=cat
=> Phát hiện: [ SSRF ] (Độ tự tin: 88.46%)



: 